In [ ]:
## %pip install mujoco

In [1]:
# Import Standard libraries
from pathlib import Path
import time
import numpy as np

#Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

In [2]:
# Settings
MJCF_PATH = Path("unitree_go2/scene_indoor.xml")

CONTROL_STEP = 0.002   # seconds per control update
PRINT_EVERY = 50

if not MJCF_PATH.is_file():
    raise FileNotFoundError(f"Could not find: {MJCF_PATH.resolve()}")

model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data = mujoco.MjData(model)

print("Loaded:", MJCF_PATH.resolve())
print("Actuators:", model.nu)
print("Sensors:", model.nsensor)

Loaded: /mnt/c/Users/Raged Rhombus/Desktop/Projects/Go2-Learning/unitree_go2/scene_indoor.xml
Actuators: 12
Sensors: 30


In [3]:
## help(mujoco.mjtObj) ## To show the structure of the "object" of our model

In [4]:
ACTUATOR_NAMES = [
    mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    for i in range(model.nu)
]

SENSOR_NAMES = [
    mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_SENSOR, i)
    for i in range(model.nsensor)
]

print("Actuators:")
for i, name in enumerate(ACTUATOR_NAMES):
    print(i, name)

print("\nSensors:")
for i, name in enumerate(SENSOR_NAMES):
    print(i, name)

Actuators:
0 FL_hip
1 FL_thigh
2 FL_calf
3 FR_hip
4 FR_thigh
5 FR_calf
6 RL_hip
7 RL_thigh
8 RL_calf
9 RR_hip
10 RR_thigh
11 RR_calf

Sensors:
0 abduction_front_left_pos
1 hip_front_left_pos
2 knee_front_left_pos
3 abduction_hind_left_pos
4 hip_hind_left_pos
5 knee_hind_left_pos
6 abduction_front_right_pos
7 hip_front_right_pos
8 knee_front_right_pos
9 abduction_hind_right_pos
10 hip_hind_right_pos
11 knee_hind_right_pos
12 abduction_front_left_vel
13 hip_front_left_vel
14 knee_front_left_vel
15 abduction_hind_left_vel
16 hip_hind_left_vel
17 knee_hind_left_vel
18 abduction_front_right_vel
19 hip_front_right_vel
20 knee_front_right_vel
21 abduction_hind_right_vel
22 hip_hind_right_vel
23 knee_hind_right_vel
24 gyro
25 accelerometer
26 orientation
27 global_position
28 global_linvel
29 global_angvel


In [5]:
# control limits defined by the model

for i, name in enumerate(ACTUATOR_NAMES):
    print(
        name,
        "limited =", bool(model.actuator_ctrllimited[i]),
        "range =", model.actuator_ctrlrange[i]
    )

FL_hip limited = True range = [-0.9472  0.9472]
FL_thigh limited = True range = [-1.4  2.5]
FL_calf limited = True range = [-2.6227  -0.84776]
FR_hip limited = True range = [-0.9472  0.9472]
FR_thigh limited = True range = [-1.4  2.5]
FR_calf limited = True range = [-2.6227  -0.84776]
RL_hip limited = True range = [-0.9472  0.9472]
RL_thigh limited = True range = [-1.4  2.5]
RL_calf limited = True range = [-2.6227  -0.84776]
RR_hip limited = True range = [-0.9472  0.9472]
RR_thigh limited = True range = [-1.4  2.5]
RR_calf limited = True range = [-2.6227  -0.84776]


In [6]:
for i, name in enumerate(ACTUATOR_NAMES):
    print(
        name,
        "transmission:", model.actuator_trntype[i],
        "dynamics:", model.actuator_dyntype[i],
        "gain:", model.actuator_gainprm[i],
        "bias:", model.actuator_biasprm[i],
        "range:", model.actuator_ctrlrange[i],
    )

FL_hip transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-0.9472  0.9472]
FL_thigh transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-1.4  2.5]
FL_calf transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-2.6227  -0.84776]
FR_hip transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-0.9472  0.9472]
FR_thigh transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-1.4  2.5]
FR_calf transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.

In [7]:
ACTUATOR_TO_POSITION_SENSOR = {
    "FL_hip":   "abduction_front_left_pos",
    "FL_thigh": "hip_front_left_pos",
    "FL_calf":  "knee_front_left_pos",

    "FR_hip":   "abduction_front_right_pos",
    "FR_thigh": "hip_front_right_pos",
    "FR_calf":  "knee_front_right_pos",

    "RL_hip":   "abduction_hind_left_pos",
    "RL_thigh": "hip_hind_left_pos",
    "RL_calf":  "knee_hind_left_pos",

    "RR_hip":   "abduction_hind_right_pos",
    "RR_thigh": "hip_hind_right_pos",
    "RR_calf":  "knee_hind_right_pos",
}

In [8]:
home_id = mujoco.mj_name2id(
    model, mujoco.mjtObj.mjOBJ_KEY, "home"
)

mujoco.mj_resetDataKeyframe(model, data, home_id)
mujoco.mj_forward(model, data)

home_ctrl = data.ctrl.copy()

In [9]:
# Separate data used only for calculating joint angles.
# The simulated robot remains controlled through data.ctrl.
ik_data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, ik_data, home_id)
mujoco.mj_forward(model, ik_data)

legs = {}
for leg in ("FL", "FR", "RL", "RR"):
    names = [f"{leg}_{part}" for part in ("hip", "thigh", "calf")]
    joint_ids = [model.joint(name + "_joint").id for name in names]
    actuator_ids = np.array([model.actuator(name).id for name in names])
    site_id = model.site(f"{leg}_foot").id

    legs[leg] = {
        "site": site_id,
        "qpos": model.jnt_qposadr[joint_ids],
        "dof": model.jnt_dofadr[joint_ids],
        "act": actuator_ids,
        "foot_home": ik_data.site_xpos[site_id].copy(),
        "limits": model.actuator_ctrlrange[actuator_ids].copy(),
    }

## Forward-speed PID experiment

Run cells from the top after restarting the kernel. This simulation uses world +X
as forward and assumes the initial heading is maintained; heading and balance
feedback are not implemented. The position actuators already provide joint PD
feedback. The outer speed PID changes foot travel, then inverse kinematics
produces joint-angle commands. No external push is applied to the robot.

The original gait has not demonstrated reliable forward walking. This addition
implements the feedback experiment; it does not establish that the gait works.
First set `USE_PID = False` and compare fixed `STEP_LENGTH` values. Only tune PID
after the robot stays upright and increasing stride reliably increases speed.
Start with P only, then a small I if needed; D is optional. Defaults leave I/D zero.
The 8 cm stride cap is an experimental bound, not a guarantee of reachability;
the existing IK residual check still rejects unreachable targets.

Set `SHOW_VIEWER = False` for headless runs. Distance excludes the initial settling
period. `history` holds measurements at each walking control update. Run the next
cell to export them; each simulation rerun clears previous measurements.


In [10]:
# Forward-speed control (world +X; robot starts facing +X).
# These gains are starting values, not validated locomotion tuning.
USE_PID = True       # False: test the original fixed-stride gait first.
SHOW_VIEWER = True   # False: repeatable headless simulation.
TARGET_SPEED = 0.05  # m/s
KP, KI, KD = 0.20, 0.0, 0.0  # Start P-only; add I/D after gait validation.
MIN_STEP_LENGTH, MAX_STEP_LENGTH = 0.0, 0.08  # metres
SPEED_FILTER_TIME = 0.15  # seconds
SETTLE_TIME, RUN_TIME = 2.0, 10.0
CONTROL_STEP = 0.01


class SpeedPID:
    """Speed error -> stride correction, with filtered D and anti-windup.

    P gain: s; I gain: dimensionless; D gain: s^2.
    Integral is stored as its contribution to the output (metres).
    """
    def __init__(self, kp, ki, kd, lower, upper, filter_time):
        if min(kp, ki, kd, filter_time) < 0 or lower >= upper:
            raise ValueError("Invalid PID gains, filter time, or limits")
        self.kp, self.ki, self.kd = kp, ki, kd
        self.lower, self.upper = lower, upper
        self.filter_time = filter_time
        self.integral = 0.0
        self.filtered_speed = None

    def update(self, target, measured, dt, nominal, integrate=True):
        if dt <= 0 or not np.all(np.isfinite([target, measured, dt, nominal])):
            raise ValueError("PID requires finite inputs and positive dt")
        previous = self.filtered_speed
        if previous is None:
            previous = measured
        alpha = dt / (self.filter_time + dt)
        self.filtered_speed = previous + alpha * (measured - previous)
        error = target - self.filtered_speed
        # Derivative on measurement avoids a kick when target speed changes.
        derivative = -(self.filtered_speed - previous) / dt
        delta_i = self.ki * error * dt if integrate else 0.0
        trial = nominal + self.kp * error + self.integral + delta_i + self.kd * derivative
        # Reject integration that would push further into saturation.
        if not ((trial > self.upper and delta_i > 0) or
                (trial < self.lower and delta_i < 0)):
            self.integral = float(np.clip(self.integral + delta_i,
                                          -(self.upper-self.lower),
                                          self.upper-self.lower))
        raw = nominal + self.kp * error + self.integral + self.kd * derivative
        return float(np.clip(raw, self.lower, self.upper))


In [11]:
PERIOD = 0.8          # Seconds per complete gait cycle
STEP_LENGTH = 0.04   # 4 cm of front-to-back foot travel
STEP_HEIGHT = 0.025  # 2.5 cm lift
STANCE_FRACTION = 0.65  # Overlap gives time with all four feet down

# Diagonal pairs share the same phase.
PHASE = {"FL": 0.0, "RR": 0.0, "FR": 0.5, "RL": 0.5}


def foot_offset(phase, step_length):
    if phase < STANCE_FRACTION:
        s = phase / STANCE_FRACTION
        x = step_length * (0.5 - s)
        z = 0.0
    else:
        s = (phase - STANCE_FRACTION) / (1 - STANCE_FRACTION)
        smooth = 3 * s**2 - 2 * s**3
        x = step_length * (smooth - 0.5)
        z = STEP_HEIGHT * np.sin(np.pi * s)**2

    return np.array([x, 0.0, z])


def solve_leg(leg, target):
    """Find joint angles for a foot target in the fixed IK reference frame."""
    info = legs[leg]
    jacobian = np.zeros((3, model.nv))

    for _ in range(15):
        mujoco.mj_forward(model, ik_data)
        error = target - ik_data.site_xpos[info["site"]]

        if np.linalg.norm(error) < 0.0005:
            break

        mujoco.mj_jacSite(
            model, ik_data, jacobian, None, info["site"]
        )
        J = jacobian[:, info["dof"]]

        # Damped inverse: convert a foot-position error into joint changes.
        dq = J.T @ np.linalg.solve(
            J @ J.T + 1e-4 * np.eye(3), error
        )

        ik_data.qpos[info["qpos"]] = np.clip(
            ik_data.qpos[info["qpos"]] + np.clip(dq, -0.1, 0.1),
            info["limits"][:, 0],
            info["limits"][:, 1],
        )

    mujoco.mj_forward(model, ik_data)
    residual = np.linalg.norm(target - ik_data.site_xpos[info["site"]])
    if residual > 0.003:
        raise RuntimeError(f"{leg}: foot target missed by {residual:.3f} m")

    return ik_data.qpos[info["qpos"]].copy()


def gait_targets(t, step_length):
    targets = home_ctrl.copy()
    blend = min(t / 1.5, 1.0)  # Gradually introduce stepping.

    for leg, info in legs.items():
        phase = (t / PERIOD + PHASE[leg]) % 1.0
        target = info["foot_home"] + blend * foot_offset(phase, step_length)
        targets[info["act"]] = solve_leg(leg, target)

    return targets

In [12]:
from contextlib import nullcontext

# Every rerun resets physics, IK, PID memory, and logs.
mujoco.mj_resetDataKeyframe(model, data, home_id)
mujoco.mj_resetDataKeyframe(model, ik_data, home_id)
mujoco.mj_forward(model, data)
mujoco.mj_forward(model, ik_data)
pid = SpeedPID(KP, KI, KD, MIN_STEP_LENGTH, MAX_STEP_LENGTH, SPEED_FILTER_TIME)
history = []
walk_start_x = None
next_control, next_report = 0.0, 1.0
last_control = None
step_length = STEP_LENGTH
filtered_speed = 0.0
dt = float(model.opt.timestep)
stop_reason = "duration reached"

context = mujoco.viewer.launch_passive(model, data) if SHOW_VIEWER else nullcontext(None)
with context as viewer:
    if viewer is not None:
        with viewer.lock():
            viewer.cam.distance = 1.5
            viewer.cam.azimuth = 135
            viewer.cam.elevation = -20
            viewer.cam.lookat[:] = data.body("base").xpos
    try:
        while data.time < SETTLE_TIME + RUN_TIME:
            if viewer is not None and not viewer.is_running():
                stop_reason = "viewer closed"
                break
            wall_start = time.perf_counter()
            if data.time + 1e-10 >= next_control:
                elapsed = CONTROL_STEP if last_control is None else float(data.time) - last_control
                last_control = float(data.time)
                next_control = float(data.time) + CONTROL_STEP
                if data.time < SETTLE_TIME:
                    data.ctrl[:] = home_ctrl
                else:
                    if walk_start_x is None:
                        walk_start_x = float(data.body("base").xpos[0])
                    walking_time = float(data.time) - SETTLE_TIME
                    speed = float(data.sensor("global_linvel").data[0])
                    requested = pid.update(TARGET_SPEED, speed, elapsed, STEP_LENGTH,
                                           integrate=walking_time >= 1.5)
                    filtered_speed = pid.filtered_speed
                    step_length = requested if USE_PID else float(np.clip(
                        STEP_LENGTH, MIN_STEP_LENGTH, MAX_STEP_LENGTH))
                    data.ctrl[:] = gait_targets(walking_time, step_length)
                    history.append(dict(time=walking_time, target_speed=TARGET_SPEED,
                        speed=speed, filtered_speed=filtered_speed, step_length=step_length,
                        distance=float(data.body("base").xpos[0]) - walk_start_x))

            mujoco.mj_step(model, data)
            mujoco.mj_forward(model, data)
            base = data.body("base")
            upright = base.xmat.reshape(3, 3)[2, 2]
            if not np.all(np.isfinite(data.qpos)):
                stop_reason = "non-finite simulation state"
                break
            if float(base.xpos[2]) < 0.15 or upright < np.cos(np.deg2rad(35)):
                stop_reason = "body dropped or tilted too far"
                break
            if viewer is not None:
                viewer.sync()
            if data.time >= next_report:
                distance = 0.0 if walk_start_x is None else float(base.xpos[0]) - walk_start_x
                print(f"t={data.time:.1f}s | target={TARGET_SPEED:.3f} m/s | "
                      f"filtered speed={filtered_speed:.3f} m/s | "
                      f"stride={step_length:.3f} m | walking distance={distance:.3f} m")
                next_report += 1.0
            if viewer is not None:
                time.sleep(max(0.0, dt - (time.perf_counter() - wall_start)))
    except KeyboardInterrupt:
        stop_reason = "interrupted"
    finally:
        # No further physics steps are taken after stopping.
        data.ctrl[:] = home_ctrl

print("Stopped:", stop_reason)
if walk_start_x is not None:
    distance = float(data.body("base").xpos[0]) - walk_start_x
    duration = float(data.time) - SETTLE_TIME
    print(f"After settling: {distance:.3f} m in {duration:.2f} s; "
          f"average forward speed {distance / max(duration, dt):.3f} m/s")


t=1.0s | target=0.050 m/s | filtered speed=0.000 m/s | stride=0.040 m | walking distance=0.000 m
t=2.0s | target=0.050 m/s | filtered speed=0.000 m/s | stride=0.040 m | walking distance=0.000 m
t=3.0s | target=0.050 m/s | filtered speed=-0.000 m/s | stride=0.050 m | walking distance=-0.002 m
t=4.0s | target=0.050 m/s | filtered speed=-0.010 m/s | stride=0.052 m | walking distance=-0.006 m
t=5.0s | target=0.050 m/s | filtered speed=-0.000 m/s | stride=0.050 m | walking distance=-0.011 m
t=6.0s | target=0.050 m/s | filtered speed=-0.009 m/s | stride=0.052 m | walking distance=-0.014 m
t=7.0s | target=0.050 m/s | filtered speed=-0.000 m/s | stride=0.050 m | walking distance=-0.018 m
t=8.0s | target=0.050 m/s | filtered speed=-0.009 m/s | stride=0.052 m | walking distance=-0.021 m
t=9.0s | target=0.050 m/s | filtered speed=-0.000 m/s | stride=0.050 m | walking distance=-0.026 m
t=10.0s | target=0.050 m/s | filtered speed=-0.005 m/s | stride=0.051 m | walking distance=-0.028 m
t=11.0s | tar

In [13]:
# Optional: save the latest run for comparing gait/PID settings.
import csv
if history:
    log_path = Path("forward_pid_log.csv")
    with log_path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    print("Saved:", log_path.resolve())
else:
    print("No walking samples. Run the simulation first.")


Saved: /mnt/c/Users/Raged Rhombus/Desktop/Projects/Go2-Learning/forward_pid_log.csv
